# XFormer LSTM Standalone - Local RTX 5060 Ti Training, No Optuna

End-to-end local training pipeline for the transformer differential-protection LSTM with the full visualization/evaluation suite, but without Optuna hyperparameter tuning.

This notebook goes directly from preprocessing to fixed-configuration cross-validation and final evaluation.

Configured for:

- Processed feature file: `datasets/LSTM_Features_Combined_20260601_024803.mat`
- Raw dataset file: `datasets/StressTestDataset_20260531_115NEW.mat`
- Feature tensor target: `X_LSTM = (N, 1569, 9)`
- Binary target: `Y_LSTM`
- Five-class scenario target: `Y_class`


## 0. Local Environment


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r'F:\Downloads\Transformer Thesis')
print(f'Project root: {PROJECT_ROOT}')
print('Use the kernel: Python (XFormer LSTM Local RTX 5060 Ti)')


## 1. Imports, Seeds & Configuration


In [ ]:
import os, json, warnings, gc
from pathlib import Path
from datetime import datetime

import scipy.io, h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, SubsetRandomSampler

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, classification_report, roc_auc_score
)

SEED = 42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); np.random.seed(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark     = True
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

EXPECTED_TIMESTEPS = 1569
EXPECTED_FEATURES  = 9
FEATURE_NAMES = ['E_D1_A','E_D2_A','E_A2_A','E_D1_B','E_D2_B','E_A2_B','E_D1_C','E_D2_C','E_A2_C']
CLASS_NAMES = {1:'Normal', 2:'Inrush', 3:'Internal', 4:'External', 5:'Unknown'}

PROJECT_ROOT = Path(r'F:\Downloads\Transformer Thesis')
DATASET_DIR = PROJECT_ROOT / 'datasets'
PROCESSED_FILE = DATASET_DIR / 'LSTM_Features_Combined_20260601_024803.mat'
RAW_FILE = DATASET_DIR / 'StressTestDataset_20260531_115NEW.mat'

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
SAVE_DIR = PROJECT_ROOT / 'runs' / f'Run_{RUN_TAG}'

for required_path in [PROJECT_ROOT, DATASET_DIR, PROCESSED_FILE, RAW_FILE]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}  |  Run: {RUN_TAG}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM   : {props.total_memory / 1024**3:.2f} GB')
print(f'Features file: {PROCESSED_FILE.name}')
print(f'Raw file     : {RAW_FILE.name}')


## 2. Data Loading Utilities


In [ ]:
def decode_matlab_string(data):
    arr = np.array(data).flatten()
    if arr.dtype.kind in ('u', 'i'):
        return ''.join(chr(int(c)) for c in arr if int(c) != 0)
    if arr.dtype.kind in ('S', 'U'):
        return ''.join(arr.astype(str).flatten()).strip().rstrip('\x00')
    if arr.size == 0:
        return ''
    return str(arr[0])


def extract_cell_array(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [str(x) if x is not None else '' for x in value]
    if isinstance(value, np.ndarray):
        if value.dtype == object:
            return [str(x) if x is not None else '' for x in value.flatten()]
        if value.dtype.kind in ('U', 'S'):
            return [str(x) for x in value.flatten()]
        return value.flatten().tolist()
    return [str(value)]


def get_struct_field(source, field):
    if source is None:
        return None
    if isinstance(source, dict):
        return source.get(field)
    if hasattr(source, field):
        return getattr(source, field)
    return None


def load_processed_features(filename):
    filename = Path(filename)
    print(f'Loading processed features: {filename.name}')
    try:
        data = scipy.io.loadmat(str(filename), simplify_cells=True)
        X = data['X_LSTM']
        Y_bin = np.asarray(data['Y_LSTM']).reshape(-1).astype(np.float32)
        Y_cls = np.asarray(data['Y_class']).reshape(-1).astype(np.int64) if 'Y_class' in data else None
        meta = data.get('metadata', {})
        print('  loader: scipy.io')
    except NotImplementedError:
        print('  loader: h5py v7.3')
        with h5py.File(str(filename), 'r') as f:
            X = np.array(f['X_LSTM'])
            Y_bin = np.array(f['Y_LSTM']).reshape(-1).astype(np.float32)
            Y_cls = np.array(f['Y_class']).reshape(-1).astype(np.int64) if 'Y_class' in f else None
            meta = {}
    return X, Y_bin, Y_cls, meta


def load_raw_dataset_metadata(filename):
    filename = Path(filename)
    print(f'Loading raw metadata reference: {filename.name}')
    try:
        data = scipy.io.loadmat(str(filename), simplify_cells=True)
        ds = data.get('dataset', {})
        if not isinstance(ds, dict):
            fields = [name for name in dir(ds) if not name.startswith('_')]
            ds = {name: getattr(ds, name) for name in fields}
        print('  raw metadata: scipy.io')
        return ds
    except Exception as exc:
        print(f'  raw metadata unavailable ({type(exc).__name__}: {exc})')
        return {}


def standardize_lstm_array(X, expected_timesteps=EXPECTED_TIMESTEPS, expected_features=EXPECTED_FEATURES):
    X = np.asarray(X)
    if X.ndim != 3:
        raise ValueError(f'X_LSTM must be 3-D, got shape {X.shape}')

    if X.shape[1] == expected_timesteps and X.shape[2] == expected_features:
        layout = 'N,T,F'
        X_ntf = X
    elif X.shape[0] == expected_features and X.shape[1] == expected_timesteps:
        layout = 'F,T,N'
        X_ntf = np.transpose(X, (2, 1, 0))
    elif X.shape[0] == expected_timesteps and X.shape[1] == expected_features:
        layout = 'T,F,N'
        X_ntf = np.transpose(X, (2, 0, 1))
    else:
        raise ValueError(
            f'Cannot infer LSTM layout from {X.shape}; expected N,{expected_timesteps},{expected_features} '
            f'or {expected_features},{expected_timesteps},N.'
        )

    return np.ascontiguousarray(X_ntf, dtype=np.float32), layout


def infer_sample_count(X):
    X = np.asarray(X)
    if X.ndim != 3:
        return int(X.shape[0])
    if X.shape[1] == EXPECTED_TIMESTEPS and X.shape[2] == EXPECTED_FEATURES:
        return int(X.shape[0])
    if X.shape[0] == EXPECTED_FEATURES and X.shape[1] == EXPECTED_TIMESTEPS:
        return int(X.shape[2])
    return int(X.shape[0])

print('Loaders defined.')


## 3. Load Data & Build Metadata DataFrame


In [ ]:
X_raw, Y_bin_raw, Y_cls_raw, metadata = load_processed_features(PROCESSED_FILE)
raw_metadata = load_raw_dataset_metadata(RAW_FILE)

N_file = infer_sample_count(X_raw)
print(f'Raw X shape from file: {X_raw.shape}')
print(f'Samples inferred     : {N_file}')
print(f'Y_LSTM shape         : {Y_bin_raw.shape}')
if Y_cls_raw is not None:
    print(f'Y_class shape        : {Y_cls_raw.shape}')

if len(Y_bin_raw) != N_file:
    raise ValueError(f'Y_LSTM length {len(Y_bin_raw)} does not match X sample count {N_file}')
if Y_cls_raw is not None and len(Y_cls_raw) != N_file:
    raise ValueError(f'Y_class length {len(Y_cls_raw)} does not match X sample count {N_file}')


def values_from_sources(fields, n, default=None, as_text=False):
    sources = [metadata, raw_metadata]
    for source in sources:
        for field in fields:
            value = get_struct_field(source, field)
            if value is None:
                continue
            if as_text:
                vals = extract_cell_array(value)
                vals = [str(v).strip() for v in vals]
            else:
                try:
                    vals = np.asarray(value).reshape(-1).tolist()
                except Exception:
                    vals = extract_cell_array(value)
            if len(vals) == n:
                if as_text and not any(vals):
                    continue
                return vals
    return [default] * n

scenario_vals = values_from_sources(['scenarioName', 'scenario_name', 'scenario', 'zone'], N_file, '', as_text=True)
zone_vals = values_from_sources(['zone', 'scenarioName', 'scenario_name', 'scenario'], N_file, '', as_text=True)
fault_vals = values_from_sources(['faultType', 'fault_type', 'fault'], N_file, '', as_text=True)
should_trip_ref = values_from_sources(['shouldTrip', 'should_trip'], N_file, None, as_text=False)

if Y_cls_raw is not None:
    class_names = [CLASS_NAMES.get(int(c), 'Unknown') for c in Y_cls_raw]
    scenario_vals = [v if v else class_names[i] for i, v in enumerate(scenario_vals)]
    # Use Y_class as the canonical protection scenario label for plots/splits.
    zone_vals = [class_names[i] if class_names[i] != 'Unknown' else (zone_vals[i] or 'Unknown')
                 for i in range(N_file)]
else:
    scenario_vals = [v if v else 'Unknown' for v in scenario_vals]
    zone_vals = [v if v else 'Unknown' for v in zone_vals]

fault_vals = [v if v else 'Unknown' for v in fault_vals]
should_trip_clean = []
for i, value in enumerate(should_trip_ref):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        should_trip_clean.append(int(Y_bin_raw[i]))
    else:
        should_trip_clean.append(int(value))

label_mismatch = int(np.sum(np.asarray(should_trip_clean, dtype=int) != Y_bin_raw.astype(int)))
if label_mismatch:
    print(f'Warning: raw shouldTrip differs from Y_LSTM for {label_mismatch} samples; training will use Y_LSTM.')
else:
    print('Raw shouldTrip check: aligned with Y_LSTM or filled from Y_LSTM.')

df = pd.DataFrame({
    'sample_id': range(N_file),
    'scenario_name': scenario_vals,
    'zone': zone_vals,
    'fault_type': fault_vals,
    'fault_resistance': values_from_sources(['faultResistance', 'fault_resistance', 'Rf'], N_file, np.nan, as_text=False),
    'noise_level': values_from_sources(['noiseLevel', 'noise_level', 'snrDb', 'SNR'], N_file, np.nan, as_text=False),
    'inception_angle': values_from_sources(['inceptionAngle', 'inception_angle'], N_file, np.nan, as_text=False),
    'inception_time': values_from_sources(['inceptionTime', 'inception_time'], N_file, np.nan, as_text=False),
    'should_trip': Y_bin_raw.astype(int),
})
if Y_cls_raw is not None:
    df['y_class'] = Y_cls_raw.astype(int)
    df['class_name'] = [CLASS_NAMES.get(int(c), 'Unknown') for c in Y_cls_raw]

print('\nScenario distribution:')
print(df['scenario_name'].value_counts(dropna=False).head(20))
print('\nBinary label distribution:')
print(df['should_trip'].value_counts().rename(index={0:'No Trip', 1:'Trip'}))
print('\nNoise levels:')
print(df['noise_level'].value_counts(dropna=False).sort_index().head(20))


## 4. Preprocessing and Stratified Train / Test Split

The current feature file is already saved as `(N, T, F) = (14128, 1569, 9)`. The helper still accepts older MATLAB-style `(F, T, N)` files, so this notebook remains backward compatible.


In [ ]:
X_np, detected_layout = standardize_lstm_array(X_raw)
print(f'Detected X_LSTM layout: {detected_layout}')

X_tensor = torch.from_numpy(X_np)
Y_tensor = torch.tensor(Y_bin_raw, dtype=torch.float32)
Yc_tensor = torch.tensor(Y_cls_raw, dtype=torch.long) if Y_cls_raw is not None else None

# Release the original float64 MATLAB array after converting to float32.
del X_raw, X_np
gc.collect()

N, T, F = X_tensor.shape
print(f'X shape: ({N}, {T}, {F})')
assert T == EXPECTED_TIMESTEPS, f'Timestep mismatch: {T} != {EXPECTED_TIMESTEPS}'
assert F == EXPECTED_FEATURES,  f'Feature mismatch: {F} != {EXPECTED_FEATURES}'
assert len(Y_tensor) == N,      f'Label mismatch: {len(Y_tensor)} != {N}'
print('Shape assertions passed')

n_pos = int(Y_tensor.sum()); n_neg = N - n_pos
pos_weight_val = n_neg / max(n_pos, 1)
print(f'\nClass balance  - Positive: {n_pos}  Negative: {n_neg}')
print(f'pos_weight     = {pos_weight_val:.3f}')

Y_strat = Y_tensor.numpy().astype(int)
train_idx, test_idx = train_test_split(
    np.arange(N), test_size=0.20, stratify=Y_strat, random_state=SEED
)

X_tr = X_tensor[train_idx]; Y_tr = Y_tensor[train_idx]
X_te = X_tensor[test_idx];  Y_te = Y_tensor[test_idx]
Yc_tr = Yc_tensor[train_idx] if Yc_tensor is not None else None
Yc_te = Yc_tensor[test_idx]  if Yc_tensor is not None else None

df_train = df.iloc[train_idx].copy().reset_index(drop=True)
df_test  = df.iloc[test_idx ].copy().reset_index(drop=True)

print(f'\nTrain: {len(train_idx)}  |  Test: {len(test_idx)}')
print(f'Train positive rate: {Y_tr.mean().item()*100:.1f}%')
print(f'Test  positive rate: {Y_te.mean().item()*100:.1f}%')


## 5. Model Architecture
Bidirectional LSTM with LayerNorm and attention pooling.  
Forward pass returns a **raw logit** — use `BCEWithLogitsLoss` for training, `torch.sigmoid` at inference.


In [ ]:
class TransformerProtectionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.3, bidirectional=True):
        super().__init__()
        D = 2 if bidirectional else 1
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        self.layer_norm = nn.LayerNorm(hidden_size * D)
        self.attention  = nn.Linear(hidden_size * D, 1)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * D, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out     = self.layer_norm(out)
        w       = torch.softmax(self.attention(out), dim=1)
        ctx     = (w * out).sum(dim=1)
        return self.classifier(ctx).squeeze(-1)

print('Model class defined.')

## 6. Training & Validation Helpers


In [ ]:
def train_epoch(model, loader, criterion, optimizer, grad_clip):
    model.train()
    total, preds_all, labels_all = 0.0, [], []
    for Xb, Yb in loader:
        Xb, Yb = Xb.to(device), Yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(Xb), Yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total += loss.item()
        preds_all .extend((torch.sigmoid(model(Xb).detach()) > 0.5).float().cpu().numpy())
        labels_all.extend(Yb.cpu().numpy())
    return total / len(loader), accuracy_score(labels_all, preds_all)

def evaluate(model, loader, criterion):
    model.eval()
    total, preds_all, labels_all, probs_all = 0.0, [], [], []
    with torch.no_grad():
        for Xb, Yb in loader:
            Xb, Yb = Xb.to(device), Yb.to(device)
            logits = model(Xb)
            total += criterion(logits, Yb).item()
            probs = torch.sigmoid(logits)
            probs_all .extend(probs.cpu().numpy())
            preds_all .extend((probs > 0.5).float().cpu().numpy())
            labels_all.extend(Yb.cpu().numpy())
    la, pa, pr = np.array(labels_all), np.array(preds_all), np.array(probs_all)
    return dict(
        loss=total/len(loader),
        accuracy =accuracy_score (la, pa),
        precision=precision_score(la, pa, zero_division=0),
        recall   =recall_score   (la, pa, zero_division=0),
        f1       =f1_score       (la, pa, zero_division=0),
        auc_roc  =roc_auc_score  (la, pr) if len(np.unique(la))>1 else float('nan'),
    ), pa, la, pr

print('Helpers defined.')

## 7. Fixed Hyperparameter Configuration

Optuna is skipped in this version. The model uses a deterministic fixed configuration suitable for local RTX 5060 Ti training.


In [ ]:
CFG = dict(
    input_size    = EXPECTED_FEATURES,
    hidden_size   = 128,
    num_layers    = 2,
    dropout       = 0.30,
    bidirectional = True,
    lr            = 1e-3,
    batch_size    = 32,
    weight_decay  = 1e-4,
    max_epochs    = 40,
    patience      = 8,
    grad_clip     = 1.0,
    k_folds       = 5,
)

TRAINING_MODE = 'fixed_no_optuna'


def build_model():
    return TransformerProtectionLSTM(
        input_size    = CFG['input_size'],
        hidden_size   = CFG['hidden_size'],
        num_layers    = CFG['num_layers'],
        dropout       = CFG['dropout'],
        bidirectional = CFG['bidirectional'],
    ).to(device)

n_params_model = sum(p.numel() for p in build_model().parameters() if p.requires_grad)

print('='*55)
print('FIXED TRAINING CONFIGURATION - NO OPTUNA')
print('='*55)
for k, v in CFG.items():
    print(f'  {k:<20}: {v}')
print(f'  {"model_params":<20}: {n_params_model:,}')
print('\nTo make a quick smoke test, temporarily set max_epochs=3 and k_folds=2.')


## 8. Stratified K-Fold Cross-Validation with Fixed Config
Now runs directly with the fixed no-Optuna hyperparameter configuration.


In [ ]:
print('STRATIFIED K-FOLD CROSS-VALIDATION')
print(f'  Device     : {device}')
print(f'  K-Folds    : {CFG["k_folds"]}')
print(f'  Max epochs : {CFG["max_epochs"]}')
print(f'  Patience   : {CFG["patience"]}')
print()

skf           = StratifiedKFold(n_splits=CFG['k_folds'], shuffle=True, random_state=SEED)
train_dataset = TensorDataset(X_tr, Y_tr)
Y_tr_np       = Y_tr.numpy().astype(int)

fold_results = []
best_models  = []

for fold, (tr_ids, val_ids) in enumerate(skf.split(np.zeros(len(Y_tr_np)), Y_tr_np)):
    print('='*65)
    print(f'FOLD {fold+1}/{CFG["k_folds"]}  —  train {len(tr_ids)}, val {len(val_ids)}')
    print('='*65)

    tr_loader  = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                            sampler=SubsetRandomSampler(tr_ids))
    val_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                            sampler=SubsetRandomSampler(val_ids))

    model     = build_model()
    pw        = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG['max_epochs'], eta_min=1e-5
    )

    best_val_loss, best_state = float('inf'), None
    patience_cnt = 0
    history = {'train_loss':[], 'val_loss':[], 'val_acc':[], 'val_auc':[]}

    pbar = tqdm(range(CFG['max_epochs']), desc=f'Fold {fold+1}')
    for epoch in pbar:
        tr_loss, _ = train_epoch(model, tr_loader, criterion, optimizer, CFG['grad_clip'])
        val_m, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'  ].append(val_m['loss'])
        history['val_acc'   ].append(val_m['accuracy'])
        history['val_auc'   ].append(val_m['auc_roc'])

        pbar.set_postfix({
            'tr_loss': f"{tr_loss:.4f}",
            'val_loss': f"{val_m['loss']:.4f}",
            'F1' : f"{val_m['f1']:.4f}",
            'AUC': f"{val_m['auc_roc']:.4f}",
        })

        if val_m['loss'] < best_val_loss:
            best_val_loss = val_m['loss']
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt  = 0
        else:
            patience_cnt += 1
            if patience_cnt >= CFG['patience']:
                print(f'\n  Early stopping at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    best_models.append(model)

    final_m, _, _, _ = evaluate(model, val_loader, criterion)
    fold_results.append({**{'fold': fold+1, 'history': history}, **final_m})
    print(f'\n  Fold {fold+1}  |  Loss {final_m["loss"]:.4f}  '
          f'Acc {final_m["accuracy"]:.4f}  F1 {final_m["f1"]:.4f}  '
          f'AUC {final_m["auc_roc"]:.4f}\n')

## 9. Cross-Validation Summary & Model Selection


In [ ]:
def format_dataframe(df, func):
    if hasattr(df, 'map'):
        return df.map(func)
    return df.applymap(func)

cv_df = pd.DataFrame([{
    'Fold':      r['fold'],
    'Val Loss':  r['loss'],
    'Accuracy':  r['accuracy'],
    'Precision': r['precision'],
    'Recall':    r['recall'],
    'F1':        r['f1'],
    'AUC-ROC':   r['auc_roc'],
} for r in fold_results])

print(cv_df.to_string(index=False, float_format='{:.4f}'.format))
print('-'*60)
stats = cv_df.iloc[:,1:].agg(['mean','std'])
print(format_dataframe(stats, lambda x: f'{x:.4f}'))

best_fold_idx = cv_df['AUC-ROC'].idxmax()
best_model    = best_models[best_fold_idx]
print(f'\nBest model: Fold {cv_df.loc[best_fold_idx, "Fold"]}')

# Training curves for best fold
hist = fold_results[best_fold_idx]['history']
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist['train_loss'], label='Train'); axes[0].plot(hist['val_loss'], label='Val')
axes[0].set_title(f'Loss — Fold {best_fold_idx+1}'); axes[0].legend()
axes[1].plot(hist['val_acc'], color='green'); axes[1].set_title('Val Accuracy'); axes[1].set_ylim([0,1.05])
axes[2].plot(hist['val_auc'], color='darkorange'); axes[2].set_title('Val AUC-ROC'); axes[2].set_ylim([0,1.05])
for ax in axes: ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 10. Final Test Set Evaluation


In [ ]:
print('='*65)
print('FINAL TEST SET EVALUATION')
print('='*65)

te_loader  = DataLoader(TensorDataset(X_te, Y_te), batch_size=CFG['batch_size'], shuffle=False)
pw         = torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pw)

te_metrics, te_preds, te_labels, te_probs = evaluate(best_model, te_loader, criterion)
for k, v in te_metrics.items(): print(f'  {k:<12}: {v:.4f}')

cm = confusion_matrix(te_labels, te_preds)
print(f'\n  Confusion Matrix:')
print(f'    TN {cm[0,0]:5d}   FP {cm[0,1]:5d}')
print(f'    FN {cm[1,0]:5d}   TP {cm[1,1]:5d}')
print('\n', classification_report(te_labels, te_preds,
      target_names=['No Trip','Trip'], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Trip','Trip'], yticklabels=['No Trip','Trip'])
axes[0].set_title('Test Set — Confusion Matrix')

fpr, tpr, _ = roc_curve(te_labels, te_probs)
axes[1].plot(fpr, tpr, lw=2, color='darkorange', label=f'AUC = {auc(fpr,tpr):.4f}')
axes[1].plot([0,1],[0,1],'--',color='navy')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve — Test Set')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 10b. Threshold Analysis — Precision · Recall · Youden J

The default 0.5 threshold is rarely optimal on imbalanced data.  
- **Youden J** maximises `TPR − FPR` (balanced sensitivity + specificity).  
- **F1-optimal** maximises the harmonic mean of precision and recall.  
- **Precision-Recall curve** is preferred over ROC when positives are rare.


In [ ]:
# ── Threshold sweep ─────────────────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve, average_precision_score

fpr_t, tpr_t, thresh_roc = roc_curve(te_labels, te_probs)
prec_t, rec_t, thresh_pr  = precision_recall_curve(te_labels, te_probs)
ap_score = average_precision_score(te_labels, te_probs)

# Youden J
j_scores  = tpr_t - fpr_t
best_j_idx  = np.argmax(j_scores)
best_j_thr  = thresh_roc[best_j_idx]

# F1-optimal threshold
f1_scores = np.where((prec_t+rec_t)>0, 2*prec_t*rec_t/(prec_t+rec_t), 0)
best_f1_idx = np.argmax(f1_scores[:-1])          # last entry has no threshold
best_f1_thr = thresh_pr[best_f1_idx]

print(f'Youden-J optimal threshold : {best_j_thr:.4f}  (TPR={tpr_t[best_j_idx]:.4f}, FPR={fpr_t[best_j_idx]:.4f})')
print(f'F1-optimal threshold        : {best_f1_thr:.4f}  (P={prec_t[best_f1_idx]:.4f}, R={rec_t[best_f1_idx]:.4f})')
print(f'Average Precision (AP)      : {ap_score:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC with threshold markers
axes[0].plot(fpr_t, tpr_t, lw=2, color='darkorange', label=f'AUC={auc(fpr_t,tpr_t):.4f}')
axes[0].plot([0,1],[0,1],'--',color='navy',alpha=0.5)
axes[0].scatter(fpr_t[best_j_idx], tpr_t[best_j_idx], s=120, color='red',
                zorder=5, label=f'Youden J (thr={best_j_thr:.3f})')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve with Youden-J Operating Point'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Precision-Recall
axes[1].plot(rec_t, prec_t, lw=2, color='steelblue', label=f'AP={ap_score:.4f}')
axes[1].scatter(rec_t[best_f1_idx], prec_t[best_f1_idx], s=120, color='red',
                zorder=5, label=f'Best F1 (thr={best_f1_thr:.3f})')
axes[1].axhline(te_labels.mean(), ls='--', color='grey', alpha=0.6, label='Baseline (prevalence)')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'Precision-Recall Curve  (AP={ap_score:.4f})'); axes[1].legend(); axes[1].grid(alpha=0.3)

# Threshold vs P / R / F1
thrs = np.linspace(0.01, 0.99, 200)
metrics_thr = {'Precision':[], 'Recall':[], 'F1':[], 'Specificity':[]}
for thr in thrs:
    pred = (te_probs >= thr).astype(int)
    tp = ((pred==1)&(te_labels==1)).sum(); fp = ((pred==1)&(te_labels==0)).sum()
    tn = ((pred==0)&(te_labels==0)).sum(); fn = ((pred==0)&(te_labels==1)).sum()
    metrics_thr['Precision'].append(tp/(tp+fp) if (tp+fp)>0 else 0)
    metrics_thr['Recall'   ].append(tp/(tp+fn) if (tp+fn)>0 else 0)
    metrics_thr['F1'       ].append(2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else 0)
    metrics_thr['Specificity'].append(tn/(tn+fp) if (tn+fp)>0 else 0)

for name, vals in metrics_thr.items():
    axes[2].plot(thrs, vals, lw=1.8, label=name)
axes[2].axvline(0.5,       ls='--', color='k',   alpha=0.5, label='thr=0.50')
axes[2].axvline(best_j_thr,ls='--', color='red', alpha=0.7, label=f'Youden={best_j_thr:.3f}')
axes[2].axvline(best_f1_thr,ls='--',color='blue',alpha=0.7, label=f'F1-opt={best_f1_thr:.3f}')
axes[2].set_xlabel('Threshold'); axes[2].set_ylabel('Metric Value')
axes[2].set_title('Metric vs Classification Threshold'); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Re-evaluate at Youden threshold
preds_j = (te_probs >= best_j_thr).astype(float)
print(f'\n── Metrics at Youden threshold ({best_j_thr:.4f}) ──')
print(f'  Accuracy  : {accuracy_score(te_labels,preds_j):.4f}')
print(f'  Precision : {precision_score(te_labels,preds_j,zero_division=0):.4f}')
print(f'  Recall    : {recall_score(te_labels,preds_j,zero_division=0):.4f}')
print(f'  F1        : {f1_score(te_labels,preds_j,zero_division=0):.4f}')
cm_j = confusion_matrix(te_labels, preds_j)
print(f'  TN={cm_j[0,0]}  FP={cm_j[0,1]}  FN={cm_j[1,0]}  TP={cm_j[1,1]}')


## 10c. Score Distribution Analysis

The sigmoid output distribution reveals how confidently the model separates classes.  
A well-trained model shows two tight, well-separated peaks.  
Overlap in the centre indicates uncertainty near the decision boundary.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

probs_pos = te_probs[te_labels == 1]
probs_neg = te_probs[te_labels == 0]

# ── Overlapping KDE / histogram ──────────────────────────────────────────
axes[0].hist(probs_neg, bins=40, alpha=0.55, color='steelblue',  density=True, label='No-Trip (y=0)')
axes[0].hist(probs_pos, bins=40, alpha=0.55, color='tomato',     density=True, label='Trip    (y=1)')
axes[0].axvline(0.5,        ls='--', color='k',   lw=1.5, label='thr=0.50')
axes[0].axvline(best_j_thr, ls='--', color='red', lw=1.5, label=f'Youden={best_j_thr:.3f}')
axes[0].set_xlabel('Predicted Probability'); axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution by True Class'); axes[0].legend(); axes[0].grid(alpha=0.3)

# ── Cumulative score distributions (separation) ──────────────────────────
axes[1].plot(np.sort(probs_neg), np.linspace(0,1,len(probs_neg)), color='steelblue', lw=2, label='No-Trip')
axes[1].plot(np.sort(probs_pos), np.linspace(0,1,len(probs_pos)), color='tomato',    lw=2, label='Trip')
axes[1].axvline(best_j_thr, ls='--', color='red', lw=1.5, label=f'Youden={best_j_thr:.3f}')
axes[1].set_xlabel('Predicted Probability'); axes[1].set_ylabel('CDF')
axes[1].set_title('Score CDFs by Class (Separation Plot)'); axes[1].legend(); axes[1].grid(alpha=0.3)

# ── Box + swarm of probs by zone ─────────────────────────────────────────
if 'zone' in df_test.columns:
    import matplotlib.patches as mpatches
    zone_order = ['Normal','Inrush','Internal','External']
    zone_colors = {'Normal':'#58c458','Inrush':'#f5a623','Internal':'#e04040','External':'#3a7fd4'}
    bp_data  = [te_probs[(df_test['zone']==z).values] for z in zone_order if (df_test['zone']==z).sum()>0]
    bp_labels= [z for z in zone_order if (df_test['zone']==z).sum()>0]

    if bp_data: # Only plot if there is data
        bplot = axes[2].boxplot(bp_data, labels=bp_labels, patch_artist=True, notch=True, widths=0.5)
        for patch, lbl in zip(bplot['boxes'], bp_labels):
            patch.set_facecolor(zone_colors.get(lbl,'grey')); patch.set_alpha(0.6)
        axes[2].axhline(0.5,        ls='--', color='k',   lw=1.5, label='thr=0.50')
        axes[2].axhline(best_j_thr, ls='--', color='red', lw=1.5, label=f'Youden={best_j_thr:.3f}')
        axes[2].set_ylabel('Predicted Probability'); axes[2].set_title('Score Distribution by Zone')
        axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3, axis='y')
    else:
        axes[2].text(0.5,0.5,'No data available for zone-wise boxplot (all zones are "Unknown")',ha='center',transform=axes[2].transAxes)
else:
    axes[2].text(0.5,0.5,'zone column not in df_test',ha='center',transform=axes[2].transAxes)

plt.suptitle('Score Distribution Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Overlap statistic
overlap = np.mean(probs_pos < 0.5) + np.mean(probs_neg >= 0.5)
print(f'Misclassified at thr=0.50  : {overlap*100:.1f}% of samples lie on wrong side')
print(f'Mean score — Trip    : {probs_pos.mean():.4f}  ± {probs_pos.std():.4f}')
print(f'Mean score — No-Trip : {probs_neg.mean():.4f}  ± {probs_neg.std():.4f}')

## 10d. Calibration Curve (Reliability Diagram)

A perfectly calibrated model has predicted probability = empirical frequency.  
Over-confident models cluster near 0 and 1; under-confident ones cluster near 0.5.  
Expected Calibration Error (ECE) and Maximum Calibration Error (MCE) quantify deviation.


In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Reliability diagram ──────────────────────────────────────────────────
for n_bins, color, ls in [(5,'steelblue','-'), (10,'darkorange','--'), (20,'green',':')]:
    frac_pos, mean_pred = calibration_curve(te_labels, te_probs, n_bins=n_bins, strategy='uniform')
    axes[0].plot(mean_pred, frac_pos, marker='o', lw=1.8, color=color, ls=ls,
                 markersize=5, label=f'{n_bins} bins')
axes[0].plot([0,1],[0,1],'--k', lw=1.2, label='Perfect calibration')
axes[0].set_xlabel('Mean Predicted Probability'); axes[0].set_ylabel('Fraction of Positives')
axes[0].set_title('Reliability Diagram (Calibration Curve)'); axes[0].legend(); axes[0].grid(alpha=0.3)

# ── ECE / MCE per bin (10-bin) ────────────────────────────────────────────
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins+1)
ece_vals = []; mce_vals = []; bin_sizes = []
for i in range(n_bins):
    mask = (te_probs >= bin_edges[i]) & (te_probs < bin_edges[i+1])
    if mask.sum() == 0:
        ece_vals.append(0); mce_vals.append(0); bin_sizes.append(0); continue
    acc  = te_labels[mask].mean()
    conf = te_probs[mask].mean()
    gap  = abs(acc - conf)
    ece_vals.append(gap * mask.sum() / len(te_labels))
    mce_vals.append(gap)
    bin_sizes.append(mask.sum())

ECE = sum(ece_vals)
MCE = max(mce_vals)
bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2
bar_w = 0.04

bars = axes[1].bar(bin_centres, mce_vals, width=bar_w, color='tomato', alpha=0.75, label='|acc−conf| per bin')
axes[1].bar(bin_centres, np.array(ece_vals), width=bar_w, color='steelblue', alpha=0.75, label='Weighted ECE contribution')
axes[1].axhline(ECE, ls='--', color='navy',  lw=1.5, label=f'ECE={ECE:.4f}')
axes[1].axhline(MCE, ls='--', color='crimson',lw=1.5, label=f'MCE={MCE:.4f}')
axes[1].set_xlabel('Predicted Probability Bin'); axes[1].set_ylabel('Calibration Gap')
axes[1].set_title(f'ECE={ECE:.4f}  MCE={MCE:.4f}'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# ── Histogram of predicted probabilities (confidence histogram) ───────────
axes[2].hist(te_probs[te_labels==0], bins=30, alpha=0.55, color='steelblue', density=False, label='No-Trip')
axes[2].hist(te_probs[te_labels==1], bins=30, alpha=0.55, color='tomato',    density=False, label='Trip')
for bs, bc in zip(bin_sizes, bin_centres):
    if bs > 0:
        axes[2].text(bc, bs/2, str(bs), ha='center', va='center', fontsize=6.5, color='black')
axes[2].set_xlabel('Predicted Probability'); axes[2].set_ylabel('Count')
axes[2].set_title('Confidence Histogram'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Model Calibration Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'ECE (Expected Calibration Error) : {ECE:.4f}')
print(f'MCE (Maximum Calibration Error)  : {MCE:.4f}')
print('  < 0.05 → well-calibrated | 0.05–0.10 → acceptable | > 0.10 → needs temperature scaling')


## 10e. Bootstrap Confidence Intervals on Test Metrics

Point estimates of Accuracy / F1 / AUC-ROC / AP have sampling variance.  
1000 bootstrap resamples give bias-corrected 95% CIs — required for any rigorous thesis comparison.


In [ ]:
N_BOOT = 1000
rng_boot = np.random.default_rng(SEED)

boot_acc=[]; boot_f1=[]; boot_auc=[]; boot_ap=[]; boot_prec=[]; boot_rec=[]

for _ in range(N_BOOT):
    idx = rng_boot.integers(0, len(te_labels), len(te_labels))
    yl, yp, ypr = te_labels[idx], te_preds[idx], te_probs[idx]
    boot_acc .append(accuracy_score (yl, yp))
    boot_f1  .append(f1_score       (yl, yp, zero_division=0))
    boot_prec.append(precision_score(yl, yp, zero_division=0))
    boot_rec .append(recall_score   (yl, yp, zero_division=0))
    if len(np.unique(yl)) > 1:
        boot_auc.append(roc_auc_score(yl, ypr))
        boot_ap .append(average_precision_score(yl, ypr))
    else:
        boot_auc.append(np.nan); boot_ap.append(np.nan)

boot_results = {
    'Accuracy' : boot_acc,  'Precision': boot_prec,
    'Recall'   : boot_rec,  'F1'        : boot_f1,
    'AUC-ROC'  : boot_auc,  'Avg Prec' : boot_ap,
}

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

print(f'{"Metric":<12}  {"Point Est":>10}  {"95% CI":>22}  {"Std":>8}')
print('-'*58)
for k, (name, vals) in enumerate(boot_results.items()):
    v = np.array(vals); v = v[~np.isnan(v)]
    lo, hi = np.percentile(v, [2.5, 97.5])
    point = np.mean(v)
    print(f'{name:<12}  {point:>10.4f}  [{lo:.4f}, {hi:.4f}]  {v.std():>8.4f}')
    axes[k].hist(v, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
    axes[k].axvline(point, color='navy',  lw=2,   label=f'Mean={point:.4f}')
    axes[k].axvline(lo,    color='red',   lw=1.5, ls='--', label=f'95% CI [{lo:.3f},{hi:.3f}]')
    axes[k].axvline(hi,    color='red',   lw=1.5, ls='--')
    axes[k].set_title(name, fontweight='bold'); axes[k].legend(fontsize=8); axes[k].grid(alpha=0.3)

plt.suptitle(f'Bootstrap Confidence Intervals  (N={N_BOOT} resamples, 95% CI)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 10f. DET Curve & McNemar Statistical Significance Test

**DET curve** (Detection Error Tradeoff): plots Miss Rate vs False Alarm Rate on a normal-deviate scale — used in IEEE protection standards literature.  
**McNemar's test** checks whether two classifiers (here: 0.50 threshold vs Youden-optimal) make *statistically different* errors.


In [ ]:
from scipy.stats import norm
from scipy.stats import chi2 as chi2_dist

# ── DET Curve ────────────────────────────────────────────────────────────
def to_normal_deviate(p):
    p = np.clip(p, 1e-6, 1-1e-6)
    return norm.ppf(p)

miss_rate = 1 - tpr_t                          # FNR
far_rate  = fpr_t                               # FPR

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

nd_miss = to_normal_deviate(miss_rate + 1e-9)
nd_far  = to_normal_deviate(far_rate  + 1e-9)
axes[0].plot(nd_far, nd_miss, lw=2, color='darkorange', label='DWT-LSTM')
axes[0].scatter(to_normal_deviate(fpr_t[best_j_idx]+1e-9),
                to_normal_deviate(miss_rate[best_j_idx]+1e-9),
                s=120, color='red', zorder=5, label=f'Youden op. point')

ticks_pct = [0.5, 1, 2, 5, 10, 20, 40]
tick_nd   = [to_normal_deviate(p/100) for p in ticks_pct]
axes[0].set_xticks(tick_nd); axes[0].set_xticklabels([f'{p}%' for p in ticks_pct])
axes[0].set_yticks(tick_nd); axes[0].set_yticklabels([f'{p}%' for p in ticks_pct])
axes[0].set_xlabel('False Alarm Rate'); axes[0].set_ylabel('Miss Rate')
axes[0].set_title('DET Curve (normal-deviate scale)'); axes[0].legend(); axes[0].grid(alpha=0.3)

# ── McNemar's Test: thr=0.5 vs Youden ───────────────────────────────────
preds_50 = (te_probs >= 0.50).astype(int)
preds_yj = (te_probs >= best_j_thr).astype(int)

# Count discordant pairs
b = np.sum((preds_50==1) & (preds_yj==0) & (te_labels==1))   # correct w/ 0.5, wrong w/ Youden
c = np.sum((preds_50==0) & (preds_yj==1) & (te_labels==1))   # wrong w/ 0.5, correct w/ Youden
b2 = np.sum((preds_50==1) & (preds_yj==0) & (te_labels==0))
c2 = np.sum((preds_50==0) & (preds_yj==1) & (te_labels==0))
b_total = b + b2; c_total = c + c2

chi2_mc = (abs(b_total - c_total) - 1)**2 / max(b_total + c_total, 1)
p_mc    = 1 - chi2_dist.cdf(chi2_mc, 1)

print(f'McNemar test (thr=0.50 vs thr={best_j_thr:.3f}):')
print(f'  b (0.50 right, Youden wrong) = {b_total}')
print(f'  c (0.50 wrong, Youden right) = {c_total}')
print(f'  χ²(1) = {chi2_mc:.3f},  p = {p_mc:.4f}')
print(f'  {"Significant difference (p<0.05)" if p_mc<0.05 else "No significant difference (p≥0.05)"}')

# ── Per-fold metric variance (radar chart) ───────────────────────────────
metric_names = ['Accuracy','Precision','Recall','F1','AUC-ROC']
fold_vals = np.array([[r['accuracy'],r['precision'],r['recall'],r['f1'],r['auc_roc']]
                       for r in fold_results])
mu_fold = fold_vals.mean(axis=0)
sd_fold = fold_vals.std(axis=0)

angles = np.linspace(0, 2*np.pi, len(metric_names), endpoint=False).tolist()
angles += angles[:1]
mu_plot = mu_fold.tolist() + mu_fold[:1].tolist()
lo_plot = (mu_fold - sd_fold).tolist() + (mu_fold - sd_fold)[:1].tolist()
hi_plot = (mu_fold + sd_fold).tolist() + (mu_fold + sd_fold)[:1].tolist()

ax_r = fig.add_subplot(1, 2, 2, polar=True)
ax_r.plot(angles, mu_plot, 'o-', lw=2, color='steelblue', label='Mean ± 1σ')
ax_r.fill(angles, hi_plot, alpha=0.15, color='steelblue')
ax_r.fill(angles, lo_plot, alpha=0.30, color='white')
ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(metric_names, size=9)
ax_r.set_ylim(0, 1); ax_r.set_yticks([0.2,0.4,0.6,0.8,1.0])
ax_r.set_title('K-Fold CV Metric Radar\n(Mean ± 1σ)', fontweight='bold', pad=15)
for k, (fold_row, fold_obj) in enumerate(zip(fold_vals, fold_results)):
    fp = fold_row.tolist() + fold_row[:1].tolist()
    ax_r.plot(angles, fp, alpha=0.35, lw=1, label=f'Fold {fold_obj["fold"]}')
ax_r.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=7)

plt.suptitle('DET Curve · McNemar Test · CV Radar', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 10g. UMAP Embedding of LSTM Hidden States

Extracts the attention-pooled context vector (the 64-dim pre-classifier representation) for every test sample and projects to 2-D with UMAP.  
Well-separated clusters indicate the model has learned discriminative representations.  
Overlapping clusters explain misclassifications.


In [ ]:
import umap

# ── Extract representations via forward hook ─────────────────────────────
representations = []
hook_labels     = []

def hook_fn(module, input, output):
    representations.append(output.detach().cpu().numpy())

# Register hook on the classifier's first linear layer input
hook = best_model.classifier[0].register_forward_hook(hook_fn)

best_model.eval()
te_loader_full = DataLoader(TensorDataset(X_te, Y_te), batch_size=CFG['batch_size'], shuffle=False)
with torch.no_grad():
    for Xb, Yb in te_loader_full:
        Xb = Xb.to(device)
        _ = best_model(Xb)
        hook_labels.extend(Yb.numpy())
hook.remove()

reps = np.vstack(representations)
hook_labels = np.array(hook_labels)

print(f'Representation matrix: {reps.shape}')

# ── UMAP projection ──────────────────────────────────────────────────────
reducer = umap.UMAP(n_components=2, random_state=SEED, n_neighbors=15, min_dist=0.1)
emb     = reducer.fit_transform(reps)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Colour by true trip label
sc0 = axes[0].scatter(emb[hook_labels==0,0], emb[hook_labels==0,1],
                       s=12, alpha=0.5, c='steelblue', label='No-Trip (y=0)')
sc1 = axes[0].scatter(emb[hook_labels==1,0], emb[hook_labels==1,1],
                       s=12, alpha=0.5, c='tomato',    label='Trip (y=1)')
axes[0].set_title('UMAP — coloured by True Label', fontweight='bold')
axes[0].legend(markerscale=2); axes[0].set_xlabel('UMAP-1'); axes[0].set_ylabel('UMAP-2')

# Colour by predicted probability (confidence)
sc2 = axes[1].scatter(emb[:,0], emb[:,1], s=12, alpha=0.6,
                       c=te_probs, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(sc2, ax=axes[1], label='Predicted Probability')
# Mark misclassified samples
wrong = (te_preds.astype(int) != hook_labels.astype(int))
axes[1].scatter(emb[wrong,0], emb[wrong,1], s=35, marker='x',
                c='black', linewidths=0.8, alpha=0.8, label=f'Misclassified (n={wrong.sum()})')
axes[1].set_title('UMAP — coloured by Predicted Probability', fontweight='bold')
axes[1].legend(markerscale=1.5, fontsize=8)
axes[1].set_xlabel('UMAP-1'); axes[1].set_ylabel('UMAP-2')

# If zone info available, overlay on left plot
if 'zone' in df_test.columns:
    zone_colors = {'Normal':'#2ecc71','Inrush':'#f39c12','Internal':'#e74c3c','External':'#3498db'}
    for zone_name, zc in zone_colors.items():
        mask_z = (df_test['zone'].values == zone_name)
        if mask_z.sum() > 0:
            axes[0].scatter(emb[mask_z,0], emb[mask_z,1], s=5, alpha=0.3, c=zc)

plt.suptitle('UMAP of LSTM Context Vectors (Test Set)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Misclassified samples: {wrong.sum()} / {len(wrong)}  ({wrong.mean()*100:.1f}%)')


## 10h. Attention Weight Heatmaps

The temporal attention weights show *which part of the 1-second window* the model relies on.  
For fault scenarios the model should attend to the post-inception window.  
For inrush it should attend to the early high-transient region.


In [ ]:
# ── Extract attention weights ─────────────────────────────────────────────
attn_weights_store = []

def attn_hook(module, input, output):
    # output is the raw attention score before softmax (shape: B, T, 1)
    raw = output.detach().cpu()
    w   = torch.softmax(raw, dim=1).squeeze(-1)   # B, T
    attn_weights_store.append(w.numpy())

attn_handle = best_model.attention.register_forward_hook(attn_hook)

best_model.eval()
all_attn = []
with torch.no_grad():
    for Xb, _ in te_loader_full:
        attn_weights_store.clear()
        _ = best_model(Xb.to(device))
        all_attn.append(attn_weights_store[0])

attn_handle.remove()
all_attn = np.vstack(all_attn)   # (N_test, T)
print(f'Attention weights shape: {all_attn.shape}')

# ── Average attention per zone ────────────────────────────────────────────
T_steps = all_attn.shape[1]
time_ax = np.linspace(0, 1.0, T_steps)   # 1-second simulation window

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Panel 0: overall mean attention
mean_attn = all_attn.mean(axis=0)
axes[0].plot(time_ax, mean_attn, color='purple', lw=1.5)
axes[0].fill_between(time_ax, mean_attn, alpha=0.25, color='purple')
axes[0].set_title('Mean Attention (all test samples)', fontweight='bold')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Attention weight')
axes[0].grid(alpha=0.3)

# Panels 1-4: per zone (if available)
if 'zone' in df_test.columns:
    zone_map   = {'Normal':'#2ecc71','Inrush':'#f39c12','Internal':'#e74c3c','External':'#3498db'}
    for pi, (zone_name, zc) in enumerate(zone_map.items()):
        mask_z = (df_test['zone'].values == zone_name)
        if mask_z.sum() == 0: continue
        z_attn = all_attn[mask_z]
        mu_z   = z_attn.mean(axis=0)
        sd_z   = z_attn.std(axis=0)
        ax     = axes[pi+1]
        ax.plot(time_ax, mu_z, color=zc, lw=1.8, label=f'{zone_name} (n={mask_z.sum()})')
        ax.fill_between(time_ax, mu_z-sd_z, mu_z+sd_z, alpha=0.20, color=zc)
        ax.set_title(f'Attention — {zone_name}', fontweight='bold')
        ax.set_xlabel('Time (s)'); ax.set_ylabel('Attention weight')
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel 5: attention heatmap (top-30 sorted by entropy)
from scipy.stats import entropy as sp_entropy
attn_entropy = np.array([sp_entropy(w+1e-12) for w in all_attn])
top30_idx    = np.argsort(attn_entropy)[:30]    # lowest entropy = most focused
hm_data      = all_attn[top30_idx]
im = axes[5].imshow(hm_data, aspect='auto', cmap='hot',
                    extent=[0, 1.0, 0, len(top30_idx)], origin='lower')
plt.colorbar(im, ax=axes[5], label='Attention weight')
axes[5].set_xlabel('Time (s)'); axes[5].set_ylabel('Sample (sorted by focus)')
axes[5].set_title('Attention Heatmap — 30 Most Focused Samples', fontweight='bold')

plt.suptitle('Temporal Attention Weight Analysis', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 10i. Input Gradient Saliency — Feature Importance over Time

Computes `∂output/∂input` for each test sample. High absolute gradient at a given (timestep, feature) means that point strongly influences the prediction.  
Averaged across a class, this reveals which DWT energy features and which time windows drive the model's decisions.


In [ ]:
import torch

# __ Saliency maps ─────────────────────────────────────────────────────────
original_training_state = best_model.training # Store original state
best_model.train() # Set to train mode to allow backward pass with cudnn RNNs
# Temporarily disable dropout for deterministic saliency if it's not already 0.0
original_dropout = best_model.lstm.dropout
if original_dropout > 0:
    best_model.lstm.dropout = 0.0

n_sal_samples = min(200, len(X_te))    # limit for speed

sal_trip   = []   # saliency for Trip samples
sal_notrip = []   # saliency for No-Trip samples

for i in range(n_sal_samples):
    x = X_te[i:i+1].to(device).requires_grad_(True)
    logit = best_model(x)
    best_model.zero_grad()
    logit.backward()
    grad = x.grad.detach().cpu().squeeze(0).abs().numpy()   # (T, F)
    if Y_te[i].item() > 0.5:
        sal_trip.append(grad)
    else:
        sal_notrip.append(grad)

# Restore original state
if original_dropout > 0:
    best_model.lstm.dropout = original_dropout
best_model.train(original_training_state) # Restore model to its original training/eval state

sal_trip_mean   = np.mean(sal_trip,   axis=0) if sal_trip   else np.zeros((T_steps, EXPECTED_FEATURES))
sal_notrip_mean = np.mean(sal_notrip, axis=0) if sal_notrip else np.zeros((T_steps, EXPECTED_FEATURES))

fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# __ Saliency heatmaps (T × F) ─────────────────────────────────────────────
im0 = axes[0,0].imshow(sal_trip_mean.T,   aspect='auto', cmap='inferno',
                        extent=[0,1.0,0,EXPECTED_FEATURES], origin='lower')
plt.colorbar(im0, ax=axes[0,0])
axes[0,0].set_yticks(np.arange(EXPECTED_FEATURES)+0.5)
axes[0,0].set_yticklabels(FEATURE_NAMES, fontsize=8)
axes[0,0].set_xlabel('Time (s)'); axes[0,0].set_title('Saliency — Trip samples', fontweight='bold')

im1 = axes[0,1].imshow(sal_notrip_mean.T, aspect='auto', cmap='inferno',
                        extent=[0,1.0,0,EXPECTED_FEATURES], origin='lower')
plt.colorbar(im1, ax=axes[0,1])
axes[0,1].set_yticks(np.arange(EXPECTED_FEATURES)+0.5)
axes[0,1].set_yticklabels(FEATURE_NAMES, fontsize=8)
axes[0,1].set_xlabel('Time (s)'); axes[0,1].set_title('Saliency — No-Trip samples', fontweight='bold')

diff_sal = sal_trip_mean - sal_notrip_mean
vmax_d = np.abs(diff_sal).max()
im2 = axes[0,2].imshow(diff_sal.T, aspect='auto', cmap='RdBu_r', vmin=-vmax_d, vmax=vmax_d,
                        extent=[0,1.0,0,EXPECTED_FEATURES], origin='lower')
plt.colorbar(im2, ax=axes[0,2])
axes[0,2].set_yticks(np.arange(EXPECTED_FEATURES)+0.5)
axes[0,2].set_yticklabels(FEATURE_NAMES, fontsize=8)
axes[0,2].set_xlabel('Time (s)')
axes[0,2].set_title('Saliency Difference (Trip − No-Trip)', fontweight='bold')

# __ Per-feature importance (marginalise over time) ────────────────────────
feat_imp_trip   = sal_trip_mean.mean(axis=0)
feat_imp_notrip = sal_notrip_mean.mean(axis=0)
feat_imp_diff   = feat_imp_trip - feat_imp_notrip

x_pos = np.arange(EXPECTED_FEATURES)
axes[1,0].barh(x_pos, feat_imp_trip,   color='tomato',    alpha=0.8)
axes[1,0].set_yticks(x_pos); axes[1,0].set_yticklabels(FEATURE_NAMES, fontsize=9)
axes[1,0].set_xlabel('Mean |Gradient|'); axes[1,0].set_title('Feature Importance — Trip', fontweight='bold')
axes[1,0].grid(alpha=0.3, axis='x')

axes[1,1].barh(x_pos, feat_imp_notrip, color='steelblue', alpha=0.8)
axes[1,1].set_yticks(x_pos); axes[1,1].set_yticklabels(FEATURE_NAMES, fontsize=9)
axes[1,1].set_xlabel('Mean |Gradient|'); axes[1,1].set_title('Feature Importance — No-Trip', fontweight='bold')
axes[1,1].grid(alpha=0.3, axis='x')

cols_diff = ['tomato' if v>0 else 'steelblue' for v in feat_imp_diff]
axes[1,2].barh(x_pos, feat_imp_diff, color=cols_diff, alpha=0.8)
axes[1,2].set_yticks(x_pos); axes[1,2].set_yticklabels(FEATURE_NAMES, fontsize=9)
axes[1,2].axvline(0, color='k', lw=0.8)
axes[1,2].set_xlabel('Importance Difference (Trip − No-Trip)')
axes[1,2].set_title('Differential Feature Importance', fontweight='bold')
axes[1,2].grid(alpha=0.3, axis='x')

plt.suptitle('Input Gradient Saliency Maps', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\nTop 3 features driving TRIP predictions:')
for i in np.argsort(feat_imp_diff)[::-1][:3]:
    print(f'  {FEATURE_NAMES[i]:<12}  diff={feat_imp_diff[i]:.5f}')
print('\nTop 3 features driving NO-TRIP predictions:')
for i in np.argsort(feat_imp_diff)[:3]:
    print(f'  {FEATURE_NAMES[i]:<12}  diff={feat_imp_diff[i]:.5f}')




## 10j. Per-Zone & Per-Fault-Type Detailed Confusion Analysis


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

zone_map   = {'Normal':1,'Inrush':2,'Internal':3,'External':4}
zone_order = ['Normal','Inrush','Internal','External']

# ── Per-zone ROC curves ───────────────────────────────────────────────────
ax = axes[0,0]
zone_aucs = {}
if 'zone' in df_test.columns:
    for zone_name in zone_order:
        mask_z = (df_test['zone'].values == zone_name)
        if mask_z.sum() < 2: continue
        yl_z = te_labels[mask_z]; yp_z = te_probs[mask_z]
        if len(np.unique(yl_z)) < 2:
            ax.plot([], [], label=f'{zone_name} (n={mask_z.sum()}, single class)')
            continue
        fpr_z, tpr_z, _ = roc_curve(yl_z, yp_z)
        auc_z = roc_auc_score(yl_z, yp_z)
        zone_aucs[zone_name] = auc_z
        ax.plot(fpr_z, tpr_z, lw=2, label=f'{zone_name} AUC={auc_z:.3f}')
ax.plot([0,1],[0,1],'--k',alpha=0.4)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Per-Zone ROC Curves', fontweight='bold'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# ── Per-zone confusion matrices (normalised) ──────────────────────────────
ax = axes[0,1]
if 'zone' in df_test.columns:
    pres_zones = [z for z in zone_order if (df_test['zone'].values==z).sum()>0]
    nz = len(pres_zones)
    heat = np.full((nz, 2), np.nan)
    for zi, zone_name in enumerate(pres_zones):
        mask_z = (df_test['zone'].values == zone_name)
        yl_z   = te_labels[mask_z]; yp_z = te_preds[mask_z]
        if len(yl_z) == 0: continue
        tp = ((yp_z==1)&(yl_z==1)).sum(); fp = ((yp_z==1)&(yl_z==0)).sum()
        tn = ((yp_z==0)&(yl_z==0)).sum(); fn = ((yp_z==0)&(yl_z==1)).sum()
        tpr_z  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        tnr_z  = tn/(tn+fp) if (tn+fp)>0 else 0.0
        heat[zi] = [tpr_z, tnr_z]
    im = ax.imshow(heat, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0,1]); ax.set_xticklabels(['TPR (Sensitivity)','TNR (Specificity)'])
    ax.set_yticks(range(nz)); ax.set_yticklabels(pres_zones)
    for zi in range(nz):
        for col in range(2):
            if not np.isnan(heat[zi,col]):
                ax.text(col, zi, f'{heat[zi,col]:.3f}', ha='center', va='center',
                        fontsize=10, color='black' if 0.3<heat[zi,col]<0.7 else 'white')
    ax.set_title('Per-Zone TPR / TNR', fontweight='bold')

# ── Per-fault-type accuracy ───────────────────────────────────────────────
ax = axes[1,0]
if 'fault_type' in df_test.columns:
    ft_list = df_test['fault_type'].unique()
    ft_acc  = {}
    for ft in ft_list:
        mask_ft = (df_test['fault_type'].values == ft)
        if mask_ft.sum() == 0: continue
        ft_acc[ft] = accuracy_score(te_labels[mask_ft], te_preds[mask_ft])
    ft_sorted = sorted(ft_acc, key=ft_acc.get)
    vals = [ft_acc[k] for k in ft_sorted]
    cols = ['tomato' if v < 0.7 else ('gold' if v < 0.9 else 'steelblue') for v in vals]
    ax.barh(ft_sorted, vals, color=cols, alpha=0.85)
    ax.axvline(0.9, ls='--', color='green',  lw=1.5, label='90%')
    ax.axvline(0.7, ls='--', color='orange', lw=1.5, label='70%')
    ax.set_xlabel('Accuracy'); ax.set_title('Per-Fault-Type Accuracy', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='x')
    for i, (k, v) in enumerate(zip(ft_sorted, vals)):
        ax.text(v+0.005, i, f'{v:.3f}', va='center', fontsize=8)

# ── Fault resistance vs correctness ──────────────────────────────────────
ax = axes[1,1]
if 'fault_resistance' in df_test.columns:
    correct   = df_test['correct'].values if 'correct' in df_test.columns else (te_preds==te_labels)
    Rf_test   = df_test['fault_resistance'].values
    Rf_nonz   = Rf_test[Rf_test > 0.005]
    cor_nonz  = correct[Rf_test > 0.005]
    if len(Rf_nonz) > 0:
        bins_r = np.percentile(Rf_nonz, np.linspace(0,100,11))
        bins_r = np.unique(bins_r)
        bin_acc=[]; bin_n=[]; bin_mid=[]
        for k in range(len(bins_r)-1):
            m = (Rf_nonz>=bins_r[k]) & (Rf_nonz<bins_r[k+1])
            if m.sum()>0:
                bin_acc.append(cor_nonz[m].mean())
                bin_n.append(m.sum())
                bin_mid.append((bins_r[k]+bins_r[k+1])/2)
        bar_cols = ['tomato' if a<0.7 else 'steelblue' for a in bin_acc]
        ax.bar(range(len(bin_mid)), bin_acc, color=bar_cols, alpha=0.8, edgecolor='white')
        ax.set_xticks(range(len(bin_mid)))
        ax.set_xticklabels([f'{m:.2g}' for m in bin_mid], rotation=40, fontsize=8)
        for i,(a,nn) in enumerate(zip(bin_acc,bin_n)):
            ax.text(i,a+0.01,f'n={nn}',ha='center',fontsize=7.5)
        ax.axhline(0.9,ls='--',color='green',lw=1.5,label='90%')
        ax.set_xlabel('Fault Resistance Rf (Ω)  [bin midpoint]')
        ax.set_ylabel('Accuracy')
        ax.set_title('Accuracy vs Fault Resistance (decile bins)', fontweight='bold')
        ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

plt.suptitle('Per-Zone & Per-Fault-Type Performance Breakdown', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 11. Failure Analysis & Parameter Breakdown


In [ ]:
df_test = df_test.copy()
df_test['predicted']       = te_preds.astype(int)
df_test['prediction_prob'] = te_probs
df_test['correct']         = (df_test['should_trip'] == df_test['predicted'])

failures = df_test[~df_test['correct']].copy()
print(f'Correct: {df_test["correct"].sum()} / {len(df_test)}  ({df_test["correct"].mean()*100:.1f}%)')
print(f'Errors : {len(failures)}')

if len(failures) > 0:
    print('\nFailures by zone:');  print(failures['zone'].value_counts())
    print('\nFailures by type:');  print(failures['fault_type'].value_counts())
    print('\nFault resistance of failures:'); print(failures['fault_resistance'].describe())
else:
    print('\n✓ No failures on test set.')

res_bins   = [0, 1, 5, 20, 100]
res_labels = ['<1 Ω','1–5 Ω','5–20 Ω','20–100 Ω']
df_test['res_bin'] = pd.cut(df_test['fault_resistance'], bins=res_bins, labels=res_labels)

print('\nAccuracy by Fault Resistance:')
print(df_test.groupby('res_bin', observed=True)['correct']
      .agg(['mean','count']).rename(columns={'mean':'Accuracy','count':'N'}))

print('\nAccuracy by Zone:')
print(df_test.groupby('zone')['correct']
      .agg(['mean','count']).rename(columns={'mean':'Accuracy','count':'N'}))

## 12. 4/5-Class Breakdown


In [ ]:
zone_to_class = {'Normal':1,'Inrush':2,'Internal':3,'External':4}
df_test['true_class'] = df_test['zone'].map(zone_to_class)

print('Per-class accuracy (binary predictions mapped back to scenario class):')
for zone_name, cid in zone_to_class.items():
    mask = df_test['zone'] == zone_name
    if mask.sum() == 0: continue
    sub_acc = df_test.loc[mask,'correct'].mean()
    n = mask.sum()
    bar = '█' * int(sub_acc * 20) + '░' * (20 - int(sub_acc * 20))
    print(f'  {zone_name:<10} N={n:4d}  {bar}  {sub_acc*100:.1f}%')

## 13. Save runs


In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Saving to: {SAVE_DIR}')

ckpt_path = os.path.join(SAVE_DIR, 'transformer_protection_lstm_best_no_optuna.pth')
torch.save({
    'model_state_dict': best_model.state_dict(),
    'model_config'    : CFG,
    'schema': {
        'input_size'   : EXPECTED_FEATURES,
        'timesteps'    : EXPECTED_TIMESTEPS,
        'feature_names': FEATURE_NAMES,
        'normalization': 'log1p',
    },
    'training_mode'  : TRAINING_MODE,
    'best_threshold' : float(best_j_thr),
    'test_metrics'   : te_metrics,
    'cv_summary'     : cv_df.to_dict(),
    'run_tag'        : RUN_TAG,
}, ckpt_path)
print('Model checkpoint saved')

# Predictions + metadata
df_test.to_csv(os.path.join(SAVE_DIR, 'test_predictions.csv'), index=False)
if len(failures) > 0:
    failures.to_csv(os.path.join(SAVE_DIR, 'failure_analysis.csv'), index=False)

summary = {
    'run_tag'         : RUN_TAG,
    'training_mode'   : TRAINING_MODE,
    'cross_validation': cv_df.drop(columns='Fold')
        .agg(['mean','std'])
        .pipe(lambda _df: format_dataframe(_df, lambda x: round(x,4)))
        .to_dict(),
    'test_set': {k: round(float(v),4) for k,v in te_metrics.items()},
    'model_params': n_params_model,
    'final_config': {k: (bool(v) if isinstance(v,bool) else v) for k,v in CFG.items()},
}
with open(os.path.join(SAVE_DIR,'performance_summary_no_optuna.json'),'w') as fj:
    json.dump(summary, fj, indent=4)
print('Performance summary saved')
print(f'\nAll artefacts -> {SAVE_DIR}')


## 14. Inference on External / Stress-Test Dataset


In [ ]:
def infer_external_dataset(model_path, data_path, device):
    print("="*65)
    print("EXTERNAL DATASET INFERENCE")
    print(f"  Model: {os.path.basename(model_path)}")
    print(f"  Data : {os.path.basename(data_path)}")
    print("="*65)

    ckpt   = torch.load(model_path, map_location=device, weights_only=False)
    cfg    = ckpt["model_config"]
    schema = ckpt.get("schema", {})
    exp_F  = schema.get("input_size", EXPECTED_FEATURES)
    exp_T  = schema.get("timesteps",  EXPECTED_TIMESTEPS)

    # ── Load optimal threshold saved during training (Youden J) ────────────
    best_thresh = ckpt.get("best_threshold", None)
    if best_thresh is None:
        print("  ⚠ No best_threshold in checkpoint — will compute from external data ROC.")

    X_raw, Y_bin, Y_cls, meta = load_processed_features(data_path)
    X_np_ext, ext_layout = standardize_lstm_array(X_raw, expected_timesteps=exp_T, expected_features=exp_F)
    print(f"  Detected layout: {ext_layout}")
    X_t = torch.from_numpy(X_np_ext).to(device)
    Y_t = torch.tensor(Y_bin, dtype=torch.float32).to(device)

    N_ext, T_ext, F_ext = X_t.shape
    print(f"  Shape: ({N_ext}, {T_ext}, {F_ext})")
    if T_ext != exp_T or F_ext != exp_F:
        print(f"  ⚠ Shape mismatch (expected {exp_T}×{exp_F}). Re-run feature extractor.")
        return

    model = TransformerProtectionLSTM(
        input_size=cfg["input_size"], hidden_size=cfg["hidden_size"],
        num_layers=cfg["num_layers"], dropout=cfg["dropout"],
        bidirectional=cfg["bidirectional"]
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    loader = DataLoader(TensorDataset(X_t, Y_t), batch_size=cfg["batch_size"], shuffle=False)
    all_probs = []
    with torch.no_grad():
        for Xb, _ in loader:
            p = torch.sigmoid(model(Xb))
            all_probs.extend(p.cpu().numpy())

    y_true = Y_bin.astype(int)
    y_prob = np.array(all_probs)

    # ── Compute AUC first (threshold-independent) ──────────────────────────
    auc_val = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan")
    fpr_r, tpr_r, thresh_r = roc_curve(y_true, y_prob)

    # ── Determine threshold ────────────────────────────────────────────────
    if best_thresh is not None:
        thr = float(best_thresh)
        thr_src = f"checkpoint Youden J ({thr:.4f})"
    else:
        # Fall back: Youden J on external ROC
        j_idx = np.argmax(tpr_r - fpr_r)
        thr = float(thresh_r[j_idx])
        thr_src = f"external-data Youden J ({thr:.4f})"
    print(f"  Threshold : {thr_src}")

    y_pred = (y_prob > thr).astype(int)

    # ── Also compute metrics at fixed 0.5 for comparison ──────────────────
    y_pred_05 = (y_prob > 0.5).astype(int)

    def _metrics(yt, yp, ypr, label):
        acc  = accuracy_score(yt, yp)
        prec = precision_score(yt, yp, zero_division=0)
        rec  = recall_score(yt, yp, zero_division=0)
        f1   = f1_score(yt, yp, zero_division=0)
        auc  = roc_auc_score(yt, ypr) if len(np.unique(yt)) > 1 else float("nan")
        cm   = confusion_matrix(yt, yp)
        print(f"\n  ── {label} ──")
        print(f"  Accuracy  : {acc:.4f}")
        print(f"  Precision : {prec:.4f}")
        print(f"  Recall    : {rec:.4f}")
        print(f"  F1        : {f1:.4f}")
        print(f"  AUC-ROC   : {auc:.4f}")
        print(f"  CM: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
        return cm, acc, prec, rec, f1, auc

    cm_opt, *_ = _metrics(y_true, y_pred,    y_prob, f"Optimal threshold ({thr:.4f})")
    cm_05,  *_ = _metrics(y_true, y_pred_05, y_prob, "Fixed threshold (0.5)")

    # ── Score distribution insight ─────────────────────────────────────────
    pos_probs = y_prob[y_true == 1]
    neg_probs = y_prob[y_true == 0]
    print(f"\n  Score stats — Positive class: mean={pos_probs.mean():.4f}  median={np.median(pos_probs):.4f}  max={pos_probs.max():.4f}")
    print(f"  Score stats — Negative class: mean={neg_probs.mean():.4f}  median={np.median(neg_probs):.4f}  max={neg_probs.max():.4f}")
    frac_above = (y_prob > thr).mean()
    print(f"  Fraction predicted TRIP at thr={thr:.4f}: {frac_above:.3f}")

    # ── Figures ────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Inference: {os.path.basename(data_path)}", fontsize=14, fontweight="bold")

    # 1) Confusion matrix — optimal threshold
    sns.heatmap(cm_opt, annot=True, fmt="d", cmap="Blues", ax=axes[0,0],
                xticklabels=["No Trip","Trip"], yticklabels=["No Trip","Trip"])
    axes[0,0].set_title(f"CM — Optimal thr ({thr:.4f})")

    # 2) Confusion matrix — 0.5
    sns.heatmap(cm_05, annot=True, fmt="d", cmap="Oranges", ax=axes[0,1],
                xticklabels=["No Trip","Trip"], yticklabels=["No Trip","Trip"])
    axes[0,1].set_title("CM — Fixed thr (0.5)")

    # 3) ROC curve with both thresholds marked
    axes[0,2].plot(fpr_r, tpr_r, lw=2, color="darkorange", label=f"AUC={auc_val:.3f}")
    axes[0,2].plot([0,1],[0,1],"--",color="navy",alpha=0.4)
    j_idx_ext = np.argmax(tpr_r - fpr_r)
    axes[0,2].scatter(fpr_r[j_idx_ext], tpr_r[j_idx_ext], s=120, zorder=5,
                      color="red", label=f"Youden J ({thresh_r[j_idx_ext]:.3f})")
    # mark 0.5 on ROC
    p05_idx = np.argmin(np.abs(thresh_r - 0.5))
    axes[0,2].scatter(fpr_r[p05_idx], tpr_r[p05_idx], s=120, zorder=5,
                      marker="s", color="green", label="thr=0.5")
    axes[0,2].set_xlabel("FPR"); axes[0,2].set_ylabel("TPR")
    axes[0,2].set_title("ROC — External Dataset"); axes[0,2].legend(); axes[0,2].grid(alpha=0.3)

    # 4) Score distribution
    bins = np.linspace(0,1,41)
    axes[1,0].hist(neg_probs, bins=bins, alpha=0.6, color="steelblue",  label="No Trip (N)")
    axes[1,0].hist(pos_probs, bins=bins, alpha=0.6, color="tomato",     label="Trip (P)")
    axes[1,0].axvline(thr,  color="red",   ls="--", lw=1.5, label=f"opt thr={thr:.3f}")
    axes[1,0].axvline(0.5,  color="green", ls=":",  lw=1.5, label="thr=0.5")
    axes[1,0].set_xlabel("Predicted probability"); axes[1,0].set_ylabel("Count")
    axes[1,0].set_title("Score Distribution"); axes[1,0].legend()

    # 5) Threshold sweep: F1 / Recall / Precision / Specificity
    threshs = np.linspace(0.01, 0.99, 200)
    f1s, precs, recs, specs = [], [], [], []
    for t in threshs:
        yp = (y_prob > t).astype(int)
        f1s.append(f1_score(y_true, yp, zero_division=0))
        precs.append(precision_score(y_true, yp, zero_division=0))
        recs.append(recall_score(y_true, yp, zero_division=0))
        tn_, fp_, fn_, tp_ = confusion_matrix(y_true, yp, labels=[0,1]).ravel()
        specs.append(tn_/(tn_+fp_) if (tn_+fp_)>0 else 0.0)
    axes[1,1].plot(threshs, f1s,   label="F1",          lw=2)
    axes[1,1].plot(threshs, precs, label="Precision",   lw=1.5, ls="--")
    axes[1,1].plot(threshs, recs,  label="Recall",      lw=1.5, ls="-.")
    axes[1,1].plot(threshs, specs, label="Specificity", lw=1.5, ls=":")
    axes[1,1].axvline(thr, color="red",   ls="--", lw=1.5, label=f"opt={thr:.3f}")
    axes[1,1].axvline(0.5, color="green", ls=":",  lw=1.5, label="0.5")
    axes[1,1].set_xlabel("Threshold"); axes[1,1].set_title("Metric vs Threshold Sweep")
    axes[1,1].legend(fontsize=8); axes[1,1].grid(alpha=0.3)

    # 6) OOD score heatmap: mean predicted prob per score decile (detect calibration drift)
    decile_edges = np.percentile(y_prob, np.arange(0,101,10))
    decile_means_pos, decile_means_neg = [], []
    for lo, hi in zip(decile_edges[:-1], decile_edges[1:]):
        mask = (y_prob >= lo) & (y_prob <= hi)
        pos_frac = y_true[mask].mean() if mask.sum() > 0 else float("nan")
        decile_means_pos.append(pos_frac)
        decile_means_neg.append(1 - pos_frac if not np.isnan(pos_frac) else float("nan"))
    xs = np.arange(10)
    axes[1,2].bar(xs, decile_means_pos, color="tomato",    alpha=0.7, label="True Positive rate")
    axes[1,2].bar(xs, decile_means_neg, bottom=decile_means_pos, color="steelblue", alpha=0.7, label="True Negative rate")
    axes[1,2].set_xticks(xs)
    axes[1,2].set_xticklabels([f"D{i+1}" for i in xs], fontsize=8)
    axes[1,2].set_xlabel("Score decile (D1=lowest)"); axes[1,2].set_ylabel("Fraction")
    axes[1,2].set_title("Calibration by Score Decile"); axes[1,2].legend(fontsize=8)

    plt.tight_layout(); plt.show()


STRESS_DATA = PROCESSED_FILE
if os.path.exists(ckpt_path) and os.path.exists(STRESS_DATA):
    infer_external_dataset(ckpt_path, STRESS_DATA, device)
else:
    print("Stress-test dataset not found — skipping inference section.")
    print(f"  Expected: {STRESS_DATA}")
